# Part 1: Conceptual Walkthrough (Weather Sentences)
This step isolates how Scikit-Learn breaks text into vocabulary matrices.

In [25]:
## Importing Libraries
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Your sample sentences
sentences = [
    'The weather is sunny', 
    'The weather is partly sunny and partly cloudy.'
]

## 1. Implement CountVectorizer
`CountVectorizer` simply counts how many times each word appears in each sentence.


In [26]:
# Initialize and transform the text
count_vec = CountVectorizer()
count_vec_matrix = count_vec.fit_transform(sentences)

# Convert to a readable DataFrame using the learned vocabulary names
count_vec_df = pd.DataFrame(count_vec_matrix.toarray(), columns=count_vec.get_feature_names_out())
print("--- CountVectorizer Matrix ---")
print(count_vec_df)

--- CountVectorizer Matrix ---
   and  cloudy  is  partly  sunny  the  weather
0    0       0   1       0      1    1        1
1    1       1   1       2      1    1        1


Output View:

Words like "the", "weather", and "is" appear exactly 1 time in both rows.
"partly" shows up 2 times in the second sentence.

## 2. Implement TfidfVectorizer
`TfidfVectorizer` lowers the mathematical weight of words that show up in every document (like "the", "weather", "is") and boosts words unique to single documents (like "cloudy").

In [27]:
# Initialize and transform the text
tfidf_vec = TfidfVectorizer()
tfidf_matrix = tfidf_vec.fit_transform(sentences)

# Convert to a readable DataFrame
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_vec.get_feature_names_out())
print("\n--- TF-IDF Matrix ---")
print(tfidf_df.round(3))


--- TF-IDF Matrix ---
     and  cloudy     is  partly  sunny    the  weather
0  0.000   0.000  0.500   0.000  0.500  0.500    0.500
1  0.353   0.353  0.251   0.706  0.251  0.251    0.251


Output View:
Look at the word "weather". Its score drops significantly in sentence 2 because it carries less unique informational value.

# Part 2: Implementation on spam.csv

The standard Kaggle `spam.csv` dataset (SMS Spam Collection) loads with columns named `v1` (the label: ham/spam) and `v2` (the raw text message).


## Step 1: Load and Clean the Dataset

Load the CSV file and rename the columns to keep your code readable.

In [16]:
# ==========================================
# Part 2: Implementation on spam.csv
# ==========================================

# Step 1: Load the Dataset
df = pd.read_csv('/voc/data/spam.csv', encoding='ISO-8859-1')

# Explicitly keep only the first two columns and rename them
df = df.iloc[:, [0, 1]]
df.columns = ['Label', 'Message']

print("Dataset loaded successfully! Preview:")
print(df.head())



Dataset loaded successfully! Preview:
  Label                                            Message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


## Step 2: Split the Data

Always split your text data before feature extraction to prevent data leakage during machine learning pipeline training.


In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['Message'], 
    df['Label'], 
    test_size=0.2, 
    random_state=42
)


## Step 3: Run CountVectorizer on Spam Text

Learn the corpus vocabulary from the training set and transform both sections.


In [22]:
# Remove common English stop words ("the", "is", "and") to focus on context words
count_vec_spam = CountVectorizer(stop_words='english')

# IMPORTANT: fit_transform on train, but ONLY transform on test
X_train_cv = count_vec_spam.fit_transform(X_train)
X_test_cv = count_vec_spam.transform(X_test)

print(f"CountVectorizer Vocabulary Size: {len(count_vec_spam.vocabulary_)}")


CountVectorizer Vocabulary Size: 7472


## Step 4: Run TfidfVectorizer on Spam Text

Now extract features using TF-IDF. This penalizes generic messaging noise and accents high-value spam triggers.


In [23]:
# Initialize TF-IDF Vectorizer
tfidf_vec_spam = TfidfVectorizer(stop_words='english')

# Process the text matrices
X_train_tfidf = tfidf_vec_spam.fit_transform(X_train)
X_test_tfidf = tfidf_vec_spam.transform(X_test)

print(f"TF-IDF Vocabulary Size: {len(tfidf_vec_spam.vocabulary_)}")


TF-IDF Vocabulary Size: 7472


## Step 5: Test Model Performance (Optional Validation)

To see which extraction method works best, quickly test them both using a simple Naive Bayes model—the gold standard for spam classification.The provided code utilizes Multinomial Naive Bayes to evaluate text classification performance by comparing word count-based features (CountVectorizer) against frequency-weighted features (TF-IDF). It initializes the classifier, trains it on labeled training sets, generates predictions on unseen test data, and calculates accuracy scores to determine the superior modeling approach. 

In [24]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# Test CountVectorizer Performance
nb_cv = MultinomialNB()
nb_cv.fit(X_train_cv, y_train)
cv_preds = nb_cv.predict(X_test_cv)
print(f"Bag of Words (CountVec) Accuracy: {accuracy_score(y_test, cv_preds):.4f}")

# Test TF-IDF Vectorizer Performance
nb_tfidf = MultinomialNB()
nb_tfidf.fit(X_train_tfidf, y_train)
tfidf_preds = nb_tfidf.predict(X_test_tfidf)
print(f"TF-IDF Vectorizer Accuracy: {accuracy_score(y_test, tfidf_preds):.4f}")


Bag of Words (CountVec) Accuracy: 0.9839
TF-IDF Vectorizer Accuracy: 0.9668


An accuracy of 98.39% for CountVectorizer and 96.68% for TF-IDF shows that both techniques are highly effective for this dataset